# D-TrOPD — Colab notebook

Self-contained: installs dependencies, clones the repo, runs the test suite,
runs training, and plots the result.

**Built so far:** Phase 0 (skeleton), Phase 1 (env + networks + KL), Phase 2 (IPPO).
D-TrOPD itself is Phase 3 and is not implemented yet — asking for `train.algo: dtropd`
raises a `NotImplementedError` that names the phase.

Everything here runs CPU-only; set `--set train.device=cuda` on a T4 for the larger runs.

In [ ]:
#@title Install dependencies
!pip -q install torch numpy pyyaml pytest pettingzoo mpe2 gymnasium matplotlib

In [ ]:
#@title Clone the repository
REPO = "https://github.com/ramsinghal08/dtropd.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import os, pathlib
if not pathlib.Path("dtropd/.git").exists():
    !git clone --branch $BRANCH $REPO dtropd
os.chdir("/content/dtropd" if pathlib.Path("/content").exists() else "dtropd")
!git log --oneline -1

In [ ]:
#@title Test suite (Phase 0 acceptance: pytest green, CLI works)
!python run.py --help
!pytest -q

In [ ]:
#@title Phase 1 — closed-form Gaussian KL vs Monte Carlo
import torch
from dtropd.kl import gaussian_kl

mu_p, ls_p = torch.tensor([0.0, 0.0], dtype=torch.float64), torch.zeros(2, dtype=torch.float64)
mu_q, ls_q = torch.tensor([1.0, -0.5], dtype=torch.float64), torch.tensor([0.405, 0.2], dtype=torch.float64)

analytic = float(gaussian_kl(mu_p, ls_p, mu_q, ls_q))
g = torch.Generator().manual_seed(0)
x = mu_p + ls_p.exp() * torch.randn((1_000_000, 2), generator=g, dtype=torch.float64)
logp = torch.distributions.Normal(mu_p, ls_p.exp()).log_prob(x).sum(-1)
logq = torch.distributions.Normal(mu_q, ls_q.exp()).log_prob(x).sum(-1)
mc = float((logp - logq).mean())
print(f"analytic={analytic:.6f}  mc={mc:.6f}  rel err={abs(analytic-mc)/analytic:.4%}")

In [ ]:
#@title Phase 1 — a look at the environment
import numpy as np
from dtropd.config import EnvConfig
from dtropd.env import CooperativeCaptureEnv

cfg = EnvConfig(heterogeneous=True)
env = CooperativeCaptureEnv(cfg, seed=0)
obs = env.reset_array(seed=0)
print("agents:", cfg.n_agents, "targets:", cfg.n_targets, "obs:", obs.shape)
rng = np.random.default_rng(0)
total = 0.0
for _ in range(cfg.max_cycles):
    obs, r, term, trunc, info = env.step_array(rng.uniform(-1, 1, size=(cfg.n_agents, 2)))
    total += r[0]
print(f"random-policy episode return: {total:.2f}  |  captures on last step: {info['captures']}")
print("zone ids:", env.zone_ids(), "sensor ranges:", env.sensor_ranges())

In [ ]:
#@title CPU smoke run (< 2 minutes)
!python run.py smoke

In [ ]:
#@title Phase 2 — IPPO acceptance run (3 seeds, homogeneous env)
!python experiments/phase2_acceptance.py --seeds 0 1 2 --workers 3 --out runs/phase2

In [ ]:
#@title Learning curves from the stored metrics
from dtropd.plot import plot_learning_curves
from IPython.display import Image, display

path = plot_learning_curves(
    ["runs/phase2/seed0", "runs/phase2/seed1", "runs/phase2/seed2"],
    "figures/phase2_return.png",
    label="IPPO (homogeneous)",
)
display(Image(str(path)))
path = plot_learning_curves(
    ["runs/phase2/seed0", "runs/phase2/seed1", "runs/phase2/seed2"],
    "figures/phase2_ev.png",
    key="explained_variance",
    label="critic explained variance",
)
display(Image(str(path)))

In [ ]:
#@title Phase 3 — beta = 0 must reproduce IPPO bitwise
!python experiments/phase3_acceptance.py --seeds 0 1 2 --epochs 15

In [ ]:
#@title Calibrate the attenuation (SPEC §10 / D1)
!python run.py calibrate --config experiments/configs/dtropd_heterogeneous.yaml --epochs 10

In [ ]:
#@title Phase 4 — all baselines train end-to-end with shared hyperparameters
!python experiments/phase4_acceptance.py --seeds 0 1 2 --workers 3